<a href="https://colab.research.google.com/github/melissa-04/melisayla-biyoinformatik/blob/main/notebooks/rna-seq/mutfak/01_tam_sayim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tam sayım defteri (mutfak tarafı)

Rehberlerdeki bütün istatistik, tek bir dosyaya dayanacak: altı örneğin gen düzeyinde sayım tablosu. O tabloyu bu defterle ürettim. Bunu da sizin çalıştırmanız gerekmiyor; ama "sayılar nereden geldi" diye soran olursa cevap, satır satır burada.

Plan basit: fare transkriptomunu indirip Salmon'a bir indeks kurduruyorum; sonra altı örneğin tam FASTQ'sunu ENA'dan sırayla çekip sayıyorum ve yer kaplamasın diye her FASTQ'yu işi bitince siliyorum. Toplamda bir buçuk saati bulabiliyor. Colab oturumu ortada koparsa dert değil; defter, bitmiş örnekleri görüp atlıyor, kaldığı yerden devam ediyor.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
KOK = '/content/drive/MyDrive/melisa-ile-biyoinformatik/rna-seq'
IDX = f'{KOK}/salmon_index_vM35'
SAYIM = f'{KOK}/sayimlar'
os.makedirs(SAYIM, exist_ok=True)
print('Çalışma klasörü:', KOK)

Mounted at /content/drive
Çalışma klasörü: /content/drive/MyDrive/melisa-ile-biyoinformatik/rna-seq


## 1. Salmon'u kuruyorum

Salmon, okuma saymanın alandaki standart araçlarından; hazır derlenmiş halini indirmek, kurmanın en dertsiz yolu. Sürümü bilerek sabitliyorum (1.10.0): bir yıl sonra çalıştıran da aynı sonucu alsın.

In [ ]:
%%bash
set -e
if [ ! -x /content/salmon/bin/salmon ]; then
  echo "Salmon indiriliyor..."
  wget -q https://github.com/COMBINE-lab/salmon/releases/download/v1.10.0/salmon-1.10.0_linux_x86_64.tar.gz -O salmon.tar.gz
  test -s salmon.tar.gz || { echo "HATA: indirme başarısız"; exit 1; }
  tar -xzf salmon.tar.gz
  mv salmon-latest_linux_x86_64 /content/salmon
  rm salmon.tar.gz
fi
/content/salmon/bin/salmon --version

salmon 1.10.0


## 2. Transkriptom ve indeks

Referans olarak GENCODE'un fare transkriptomunu kullanıyorum (sürüm M35; bunu da sabitliyorum). `--gencode` bayrağı önemli: GENCODE'un uzun başlıklarını kırpıp transkript kimliğini temiz bırakıyor. İndeks kurulumu 15-20 dakika sürüyor; bittiğinde Drive'a kaydediyorum ki bir daha beklemeyeyim. Dürüstlük notu: en iyi uygulama, indekse genomu da "yem" (decoy) olarak eklemek; Colab'ın belleği buna yetmediği için yalnız transkriptomla kurdum ve bunu rehberde de söyleyeceğim.

In [ ]:
import os
if os.path.exists(f'{IDX}/info.json'):
    print('İndeks Drive\'da hazır, atlanıyor.')
else:
    !wget -q https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_mouse/release_M35/gencode.vM35.transcripts.fa.gz
    !/content/salmon/bin/salmon index --gencode -k 31 -p 2 \
        -t gencode.vM35.transcripts.fa.gz -i /content/salmon_index
    !cp -r /content/salmon_index "{IDX}"
    print('İndeks kuruldu ve Drive\'a kaydedildi.')

Version Server Response: Not Found
index ["/content/salmon_index"] did not previously exist  . . . creating it
[2026-08-31 20:59:01.514] [jLog] [warning] The salmon index is being built without any decoy sequences.  It is recommended that decoy sequence (either computed auxiliary decoy sequence or the genome of the organism) be provided during indexing. Further details can be found at https://salmon.readthedocs.io/en/latest/salmon.html#preparing-transcriptome-indices-mapping-based-mode.
[2026-08-31 20:59:01.514] [jLog] [info] building index
out : /content/salmon_index
[2026-08-31 20:59:01.514] [puff::index::jointLog] [info] Running fixFasta

[Step 1 of 4] : counting k-mers
[2026-08-31 20:59:01.548] [puff::index::jointLog] [warning] Entry with header [ENSMUST00000191703.2|ENSMUSG00000103282.2|OTTMUSG00000050290.1|OTTMUST00000127724.1|Gm37275-201|Gm37275|30|processed_pseudogene|], had length less than equal to the k-mer length of 31 (perhaps after poly-A clipping)
[2026-08-31 20:59:02.51

## 3. Transkript → gen eşlemesi

Salmon transkript düzeyinde sayar; bizim tablomuz gen düzeyinde olacak. Eşleme listesini ayrı bir yerden aramaya gerek yok, GENCODE dosyasının başlık satırlarında her transkriptin geni ve gen adı zaten yazıyor; oradan çıkarıyorum.

In [ ]:
import gzip, pandas as pd

if not os.path.exists('gencode.vM35.transcripts.fa.gz'):
    !wget -q https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_mouse/release_M35/gencode.vM35.transcripts.fa.gz

kayitlar = []
with gzip.open('gencode.vM35.transcripts.fa.gz', 'rt') as f:
    for satir in f:
        if satir.startswith('>'):
            p = satir[1:].strip().split('|')
            kayitlar.append((p[0], p[1], p[5]))
tx2gen = pd.DataFrame(kayitlar, columns=['transkript', 'gen_id', 'gen_adi'])
print(len(tx2gen), 'transkript,', tx2gen.gen_id.nunique(), 'gen')
tx2gen.head(3)

149138 transkript, 57132 gen


,transkript,gen_id,gen_adi
0,ENSMUST00000193812.2,ENSMUSG00000102693.2,4933401J01Rik
1,ENSMUST00000082908.3,ENSMUSG00000064842.3,Gm26206
2,ENSMUST00000162897.2,ENSMUSG00000051951.6,Xkr4


## 4. Altı örneği sırayla say

Döngü her örnek için aynı şeyi yapıyor: tam FASTQ'yu ENA'dan indir, Salmon'la say, sonucu Drive'a kaydet, FASTQ'yu sil. `-l A` kütüphane tipini Salmon'un kendisinin bulması demek. Daha önce bitmiş örnekler atlanıyor; oturum koparsa defteri baştan çalıştırmak yeterli.

In [ ]:
import io, requests
import os
import pandas as pd

url = ('https://www.ebi.ac.uk/ena/portal/api/filereport'
       '?accession=SRP009464&result=read_run&fields=run_accession,fastq_ftp&format=tsv')
ena = pd.read_csv(io.StringIO(requests.get(url, timeout=60).text), sep='\t')

for _, r in ena.iterrows():
    run, ftp = r['run_accession'], 'https://' + r['fastq_ftp']
    hedef = f'{SAYIM}/{run}' # Google Drive'daki hedef yol: /content/drive/.../sayimlar/SRRxxxxx

    # Dosyanın beklenen son konumda olup olmadığını kontrol et
    if os.path.exists(f'{hedef}/quant.sf'):
        print(run, 'zaten sayılmış, atlanıyor.'); continue

    print(run, 'indiriliyor...')
    # FASTQ'yu yerel /content/ dizinine indir
    !curl -sL -o /content/{run}.fastq.gz "{ftp}"

    print(run, 'sayılıyor...')
    # Salmon çıktılarını geçici yerel bir dizine yaz: /content/{run}_salmon_temp
    salmon_output_dir_local = f'/content/{run}_salmon_temp'
    !/content/salmon/bin/salmon quant -i "{IDX}" -l A -r /content/{run}.fastq.gz \
        -p 2 -o "{salmon_output_dir_local}" --no-version-check -q

    # Google Drive'daki hedef dizinin var olduğundan emin ol
    os.makedirs(hedef, exist_ok=True)

    # Salmon çıktı dizininin İÇERİĞİNİ Google Drive'daki hedefe taşı
    # Bu, iç içe klasör sorununu önleyen kritik değişikliktir
    !mv "{salmon_output_dir_local}"/* "{hedef}/"

    # Yerel geçici dosyaları temizle
    !rm /content/{run}.fastq.gz
    !rm -r "{salmon_output_dir_local}" # Boş Salmon çıktı dizinini kaldır

    print(run, 'tamam.')

SRR384978 indiriliyor...
SRR384978 sayılıyor...
-----------------------------------------
| Loading contig table | Time = 557.42 ms
-----------------------------------------
size = 881033
-----------------------------------------
| Loading contig offsets | Time = 10.436 ms
-----------------------------------------
-----------------------------------------
| Loading reference lengths | Time = 2.8205 ms
-----------------------------------------
-----------------------------------------
| Loading mphf table | Time = 359.19 ms
-----------------------------------------
size = 149666271
Number of ones: 881032
Number of ones per inventory item: 512
Inventory entries filled: 1721
-----------------------------------------
| Loading contig boundaries | Time = 699.87 ms
-----------------------------------------
size = 149666271
-----------------------------------------
| Loading sequence | Time = 127.39 ms
-----------------------------------------
size = 123235311
--------------------------------

## 5. Gen düzeyi sayım tablosu

Her örneğin quant.sf dosyasından okuma sayılarını alıp transkriptleri genlerinde topluyorum, altı örneği tek tabloda birleştiriyorum. Eşleşme oranlarını da yazdırıyorum; bir örnek diğerlerinden bariz düşükse bir terslik var demektir.

In [ ]:
import os, glob, json, gzip, subprocess
import pandas as pd
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

KOK   = '/content/drive/MyDrive/melisa-ile-biyoinformatik/rna-seq'
SAYIM = f'{KOK}/sayimlar'

# Transkript -> gen eşlemesi yoksa (oturum sıfırlandıysa) yeniden kur
if 'tx2gen' not in globals():
    fa = 'gencode.vM35.transcripts.fa.gz'
    if not os.path.exists(fa):
        subprocess.run(['wget', '-q', 'https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_mouse/release_M35/' + fa], check=True)
    kayitlar = []
    with gzip.open(fa, 'rt') as f:
        for satir in f:
            if satir.startswith('>'):
                p = satir[1:].strip().split('|')
                kayitlar.append((p[0], p[1], p[5]))
    tx2gen = pd.DataFrame(kayitlar, columns=['transkript', 'gen_id', 'gen_adi'])
    print('Eşleme kuruldu:', len(tx2gen), 'transkript')

ornek_adi = {'SRR384977':'WT_1','SRR384978':'WT_2','SRR384979':'WT_3',
             'SRR384980':'KO_1','SRR384981':'KO_2','SRR384982':'KO_3'}

def quant_bul(run):
    # Klasörün altında nerede olursa olsun quant.sf'yi bul; meta_info'su olan ve en yeni olanı seç
    adaylar = glob.glob(f'{SAYIM}/{run}/**/quant.sf', recursive=True)
    if not adaylar:
        raise FileNotFoundError(f'{run} için quant.sf bulunamadı')
    def puan(p):
        meta = os.path.exists(os.path.join(os.path.dirname(p), 'aux_info', 'meta_info.json'))
        return (meta, os.path.getmtime(p))
    return sorted(adaylar, key=puan)[-1]

tablolar = {}
for run, ad in ornek_adi.items():
    q_yol = quant_bul(run)
    print(f'{ad}: {q_yol.replace(KOK, "...")}')
    q = pd.read_csv(q_yol, sep='\t')
    q = q.merge(tx2gen, left_on='Name', right_on='transkript')
    if len(q) == 0:
        raise ValueError(f'{run}: transkript adları eşleşmedi (--gencode kullanılmamış olabilir)')
    tablolar[ad] = q.groupby(['gen_id', 'gen_adi'])['NumReads'].sum()
    meta_yol = os.path.join(os.path.dirname(q_yol), 'aux_info', 'meta_info.json')
    if os.path.exists(meta_yol):
        print('   eşleşme oranı: %', round(json.load(open(meta_yol))['percent_mapped'], 1))
    else:
        print('   eşleşme oranı: meta_info.json bulunamadı')

sayim = pd.DataFrame(tablolar).round().astype(int).reset_index()
sayim.to_csv(f'{KOK}/sayim_tablosu.csv', index=False)
print('\nKaydedildi:', f'{KOK}/sayim_tablosu.csv', '—', len(sayim), 'gen × 6 örnek')
sayim.head()

Eşleme kuruldu: 149138 transkript
WT_1: .../sayimlar/SRR384977/quant.sf
   eşleşme oranı: % 67.9
WT_2: .../sayimlar/SRR384978/SRR384978/quant.sf
   eşleşme oranı: % 53.7
WT_3: .../sayimlar/SRR384979/quant.sf
   eşleşme oranı: % 71.9
KO_1: .../sayimlar/SRR384980/quant.sf
   eşleşme oranı: % 48.8
KO_2: .../sayimlar/SRR384981/quant.sf
   eşleşme oranı: % 69.8
KO_3: .../sayimlar/SRR384982/quant.sf
   eşleşme oranı: % 53.3

Kaydedildi: /content/drive/MyDrive/melisa-ile-biyoinformatik/rna-seq/sayim_tablosu.csv — 56065 gen × 6 örnek


,gen_id,gen_adi,WT_1,WT_2,WT_3,KO_1,KO_2,KO_3
0,ENSMUSG00000000001.5,Gnai3,1419,662,2800,2182,2872,2462
1,ENSMUSG00000000003.16,Pbsn,0,0,0,0,0,0
2,ENSMUSG00000000028.16,Cdc45,1654,1099,2770,1155,1891,1299
3,ENSMUSG00000000031.19,H19,26461,23546,74207,19488,32414,45889
4,ENSMUSG00000000037.18,Scml2,37,15,82,10,20,10


## 6. İlk bakış: Klf1'in kendisi

Tablo doğru mu diye en basit kontrol: knockout yaptıkları genin kendisine bakmak. Klf1 sayıları KO örneklerde dibe vurmuş olmalı; vurmadıysa ya etiketler karışmıştır ya da bir şey ters gitmiştir.

In [ ]:
sayim[sayim.gen_adi == 'Klf1']

,gen_id,gen_adi,WT_1,WT_2,WT_3,KO_1,KO_2,KO_3
15642,ENSMUSG00000054191.10,Klf1,9339,9019,15689,1093,1690,1275
